# 📖 Notebook 9: Production Deployment — Self-Hosted, Cloud, Scaling, and Security

Taking Temporal to production requires understanding **server architecture**, **deployment options**, **worker scaling**, **security**, and **monitoring**. Running a demo locally proves the programming model; running it in production means making good operational decisions day after day.

This notebook is intentionally more conceptual and reference-oriented than earlier labs. Think of it as a field guide for the questions teams ask right before real customer traffic arrives.

## Learning Objectives

- Understand Temporal Server components: Frontend, History, Matching, and Worker service
- Compare self-hosted Temporal with Temporal Cloud
- Configure worker concurrency and scaling
- Understand mTLS and API key authentication
- Set up basic metrics with OpenTelemetry
- Use Search Attributes for workflow discoverability


## 🛠️ Setup

Start the local lab stack and install Python dependencies:

```bash
cd 03-technologies/workflow-engines/temporal
docker compose up -d
uv sync
```

- Temporal gRPC endpoint: `localhost:7233`
- Temporal Web UI: http://localhost:8080
- Select the `.venv` kernel in VS Code's kernel picker (top-right of the notebook).
- If the kernel does not appear, reload the VS Code window: `Cmd+Shift+P` → `Reload Window`.

> This notebook mixes runnable snippets with production reference material. Some examples are intentionally conceptual and need real certificates, cloud namespaces, or infrastructure before you run them.


In [ ]:
from datetime import timedelta
from pathlib import Path

from temporalio import activity, workflow
from temporalio.client import Client, TLSConfig
from temporalio.common import SearchAttributeKey
from temporalio.runtime import PrometheusConfig, Runtime, TelemetryConfig
from temporalio.worker import Worker

client = await Client.connect("localhost:7233")
print("✅ Connected to Temporal on localhost:7233")


## Temporal Server Architecture

A Temporal cluster is not just "one server". It is a set of cooperating services, each with a different job:

```
┌──────────────────────────────────────────────────────┐
│                  Temporal Cluster                    │
│                                                      │
│  ┌────────────┐  ┌────────────┐  ┌────────────┐      │
│  │  Frontend  │  │  History   │  │  Matching  │      │
│  │   (API)    │  │  (state)   │  │(task queue)│      │
│  └─────┬──────┘  └─────┬──────┘  └─────┬──────┘      │
│        │               │               │             │
│  ┌─────▼───────────────▼───────────────▼──────┐      │
│  │          Persistence Layer                 │      │
│  │  PostgreSQL / MySQL / Cassandra            │      │
│  │  + Elasticsearch (visibility)              │      │
│  └────────────────────────────────────────────┘      │
│                                                      │
│  ┌────────────┐                                      │
│  │   Web UI   │  http://localhost:8080               │
│  └────────────┘                                      │
└──────────────────────────────────────────────────────┘
```


### Frontend
The **Frontend** service is the front door. Clients and SDKs talk to it over gRPC to start workflows, send signals, complete tasks, and fetch results.

### History
The **History** service is the durable brain of Temporal. It records the event history of every workflow execution and decides what should happen next based on that history.

### Matching
The **Matching** service is the traffic controller for task queues. It matches workflow tasks and activity tasks with available workers that are polling the right queue.

### Worker service
The server-side **Worker service** handles internal background jobs inside the Temporal cluster, such as system workflows and maintenance tasks. This is different from **your application workers**, which run your workflow and activity code.


## Self-Hosted vs Temporal Cloud

| Dimension | Temporal Cloud | Self-Hosted |
|---|---|---|
| Operations | Managed | Your responsibility |
| Persistence | Managed | You manage PostgreSQL/Cassandra |
| Scaling | Managed | You scale server components |
| Upgrades | Managed | You plan schema + binary upgrades |
| Security | mTLS / API keys built-in | Full control, more responsibility |
| Cost | Service pricing | Infrastructure + ops cost |
| Best For | Teams wanting durability without ops | Teams needing full control |

A simple mental model: **Temporal Cloud lets you focus on workflows; self-hosting means you also become part Temporal operator.**

## Worker Scaling

Workers are **your** processes — Temporal does not scale them for you. The server places work onto a task queue, and your workers must be running, healthy, and numerous enough to drain that queue.

```
                task queue
                    │
        ┌───────────┼───────────┐
        ▼           ▼           ▼
    worker-1     worker-2     worker-3
```

Scale horizontally by running more worker replicas on the **same task queue**. Tune `max_concurrent_activities` and `max_concurrent_workflow_tasks` so each process uses CPU, memory, and downstream APIs responsibly instead of trying to do everything at once.


In [ ]:
@activity.defn
async def my_activity(order_id: str) -> str:
    return f"processed {order_id}"


@workflow.defn
class MyWorkflow:
    @workflow.run
    async def run(self, order_id: str) -> str:
        return await workflow.execute_activity(
            my_activity,
            order_id,
            start_to_close_timeout=timedelta(seconds=30),
        )


worker = Worker(
    client,
    task_queue="production-queue",
    workflows=[MyWorkflow],
    activities=[my_activity],
    max_concurrent_activities=100,
    max_concurrent_workflow_tasks=100,
)

print("✅ Worker configured for 'production-queue'")
print("   Tune these numbers to match your hardware and downstream limits.")


## Task Queue Isolation

A useful production pattern is separating work by **shape**. IO-heavy tasks mostly wait on networks and databases. CPU-heavy tasks spend time burning cores. If you put both on the same task queue with the same worker pool, they can hurt each other.

```
API calls, DB queries  ─────────▶  io-tasks   ─────────▶  larger worker pool
PDF rendering, ETL, ML ─────────▶  cpu-tasks  ─────────▶  smaller, CPU-focused pool
```

This isolation lets you tune concurrency, autoscaling rules, and machine sizes independently.

In [ ]:
@activity.defn
async def fetch_customer_profile(customer_id: str) -> str:
    return f"profile:{customer_id}"


@activity.defn
async def crunch_financial_report(report_id: str) -> str:
    return f"report:{report_id}"


@workflow.defn
class IOWorkflow:
    @workflow.run
    async def run(self, customer_id: str) -> str:
        return await workflow.execute_activity(
            fetch_customer_profile,
            customer_id,
            start_to_close_timeout=timedelta(seconds=30),
        )


@workflow.defn
class CPUWorkflow:
    @workflow.run
    async def run(self, report_id: str) -> str:
        return await workflow.execute_activity(
            crunch_financial_report,
            report_id,
            start_to_close_timeout=timedelta(seconds=30),
        )


io_worker = Worker(
    client,
    task_queue="io-tasks",
    workflows=[IOWorkflow],
    activities=[fetch_customer_profile],
    max_concurrent_activities=200,
    max_concurrent_workflow_tasks=100,
)

cpu_worker = Worker(
    client,
    task_queue="cpu-tasks",
    workflows=[CPUWorkflow],
    activities=[crunch_financial_report],
    max_concurrent_activities=8,
    max_concurrent_workflow_tasks=32,
)

print("✅ Defined separate worker pools for IO-heavy and CPU-heavy work")


## Security: mTLS and API Keys

Production Temporal traffic should be authenticated and encrypted. The two common patterns are:

- **mTLS**: both client and server prove their identity with certificates. This is common when you want strong machine-to-machine trust.
- **API keys**: simpler operationally, especially in Temporal Cloud, where you want a lightweight way to authenticate clients and workers.

Keep certificates and API keys in a secret manager, not in notebooks, source control, or container images.

In [ ]:
async def connect_to_temporal_cloud_example() -> Client:
    return await Client.connect(
        "your-namespace.tmprl.cloud:7233",
        namespace="your-namespace",
        tls=TLSConfig(
            client_cert=Path("client.pem").read_bytes(),
            client_private_key=Path("client.key").read_bytes(),
        ),
    )


async def connect_with_api_key_example() -> Client:
    return await Client.connect(
        "your-namespace.tmprl.cloud:7233",
        namespace="your-namespace",
        api_key="replace-with-real-api-key",
        tls=True,
    )


print("✅ Defined Temporal Cloud connection examples")
print("   Replace namespace, certificate files, and API key with real values before running them.")


## Search Attributes

Search Attributes are custom indexed fields attached to a workflow execution. They make workflows easier to find in the UI and CLI because you can search by business data instead of memorizing workflow IDs.

```
business state  ──▶  Search Attributes  ──▶  UI / CLI filters
 customer_id           CustomerId             CustomerId = 'cust-42'
 order_status          OrderStatus            OrderStatus = 'completed'
```

A beginner-friendly shorthand looks like this:

```python
workflow.upsert_search_attributes({
    "CustomerId": [customer_id],
    "OrderStatus": ["processing"],
})
```

The current Python SDK prefers **typed search attribute keys**, so the next runnable example uses that style. In production, register custom search attributes in the cluster before workflows start writing them.

In [ ]:
customer_id_key = SearchAttributeKey.for_keyword_list("CustomerId")
order_status_key = SearchAttributeKey.for_keyword_list("OrderStatus")


@workflow.defn
class OrderVisibilityWorkflow:
    @workflow.run
    async def run(self, order_id: str, customer_id: str) -> str:
        workflow.upsert_search_attributes([
            customer_id_key.value_set([customer_id]),
            order_status_key.value_set(["processing"]),
        ])

        result = await workflow.execute_activity(
            my_activity,
            order_id,
            start_to_close_timeout=timedelta(seconds=30),
        )

        workflow.upsert_search_attributes([
            order_status_key.value_set(["completed"]),
        ])
        return result


print("✅ Defined OrderVisibilityWorkflow with Search Attributes")
print("   Register CustomerId and OrderStatus in your cluster before using this in production.")


## Metrics with OpenTelemetry

Metrics give you an early warning system. A healthy production deployment usually tracks task queue lag, workflow task latency, activity failures, worker saturation, and poller health.

Temporal's Python SDK can expose Prometheus metrics through the runtime. Prometheus scrapes the endpoint, and Grafana turns those numbers into dashboards and alerts.

In [ ]:
runtime = Runtime(
    telemetry=TelemetryConfig(
        metrics=PrometheusConfig(bind_address="0.0.0.0:9090")
    )
)

metrics_client = await Client.connect("localhost:7233", runtime=runtime)
print("✅ Example client created with Prometheus metrics on 0.0.0.0:9090")
print("   Scrape that endpoint with Prometheus, then graph it in Grafana.")


## Production Checklist

- [ ] External persistence (PostgreSQL/MySQL)
- [ ] Separate visibility store
- [ ] mTLS or API key authentication
- [ ] Worker replicas (2+ per task queue)
- [ ] Resource limits on workers
- [ ] Prometheus metrics scraping
- [ ] Grafana dashboards
- [ ] Search Attributes for business fields
- [ ] Backup and restore tested
- [ ] Schema upgrade strategy
- [ ] Worker Versioning plan


## 🎓 What You Learned

- A Temporal cluster has specialized services: **Frontend** for API traffic, **History** for durable workflow state, **Matching** for task routing, and an internal **Worker service** for cluster jobs.
- **Temporal Cloud** removes most operational work, while **self-hosting** gives maximum control at the cost of running persistence, upgrades, scaling, and security yourself.
- Your **application workers are your responsibility**: scale them horizontally, tune concurrency, and isolate different workload shapes with separate task queues.
- Production readiness is not just "does the workflow run?" It also includes authentication, observability, visibility, backups, and a safe upgrade strategy.
